In [17]:
import pandas as pd
import numpy as np

In [18]:
ground_truth_df = pd.read_csv('../datasets/ground-truth.csv')
ground_truth_df

,Project,Antipattern,File,Line from,Line to,Comment
0,alexxiuccia/TrackMe,Rounding Errors,codice/TM_db/src-generated/it/trackme/jooq/gen...,71,71,toiteväärtuse andmed
1,alexxiuccia/TrackMe,Rounding Errors,codice/TM_db/src-generated/it/trackme/jooq/gen...,76,76,toiteväärtuse andmed
2,alexxiuccia/TrackMe,Rounding Errors,codice/TM_db/src-generated/it/trackme/jooq/gen...,81,81,toiteväärtuse andmed
3,alexxiuccia/TrackMe,Keyless Entry,codice/TM_db/src-generated/it/trackme/jooq/gen...,55,55,viide ricetta tabeli primaarvõtmele
4,alexxiuccia/TrackMe,Keyless Entry,codice/TM_db/src-generated/it/trackme/jooq/gen...,60,60,viide alimento tabeli primaarvõtmele
...,...,...,...,...,...,...
1557,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Ban...,61,61,select(TABLE).from(TABLE)
1558,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Cat...,115,115,selectDistinct(TABLE).from(TABLE)
1559,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Tra...,98,98,select(TABLE).from(TABLE)
1560,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Tra...,105,105,select(TABLE).from(TABLE)


In [19]:
included_antipatterns = [
    'Implicit Columns', 
    'ID Required', 
    'Keyless Entry', 
    'Beware of the Unknown', 
    'Poor Man’s Search Engine',
    'Rounding Errors',
    '31 Flavors'
]

In [20]:
filtered_ground_truth_df = ground_truth_df[ground_truth_df["Antipattern"].isin(included_antipatterns)]
filtered_ground_truth_df

,Project,Antipattern,File,Line from,Line to,Comment
0,alexxiuccia/TrackMe,Rounding Errors,codice/TM_db/src-generated/it/trackme/jooq/gen...,71,71,toiteväärtuse andmed
1,alexxiuccia/TrackMe,Rounding Errors,codice/TM_db/src-generated/it/trackme/jooq/gen...,76,76,toiteväärtuse andmed
2,alexxiuccia/TrackMe,Rounding Errors,codice/TM_db/src-generated/it/trackme/jooq/gen...,81,81,toiteväärtuse andmed
3,alexxiuccia/TrackMe,Keyless Entry,codice/TM_db/src-generated/it/trackme/jooq/gen...,55,55,viide ricetta tabeli primaarvõtmele
4,alexxiuccia/TrackMe,Keyless Entry,codice/TM_db/src-generated/it/trackme/jooq/gen...,60,60,viide alimento tabeli primaarvõtmele
...,...,...,...,...,...,...
1557,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Ban...,61,61,select(TABLE).from(TABLE)
1558,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Cat...,115,115,selectDistinct(TABLE).from(TABLE)
1559,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Tra...,98,98,select(TABLE).from(TABLE)
1560,ydeng11/Minance,Implicit Columns,src/main/java/today/ihelio/minance/service/Tra...,105,105,select(TABLE).from(TABLE)


In [21]:
iterations = 1000000

def optimize_split(df):
    print("Pre-calculating project matrix...")
    
    matrix = df[df['Antipattern'].isin(included_antipatterns)].pivot_table(index='Project', columns='Antipattern', aggfunc='size', fill_value=0)
    
    # Ensure all target columns exist (fill 0 if missing from data)
    for col in included_antipatterns:
        if col not in matrix.columns:
            matrix[col] = 0
            
    # Align columns strictly to config order
    matrix = matrix[included_antipatterns]
    
    # Convert to pure NumPy arrays for raw speed
    projects_arr = matrix.index.to_numpy()
    values_arr = matrix.to_numpy()  # Shape: (n_projects, n_antipatterns)
    
    n_projects = len(projects_arr)
    
    best_score = float('inf')
    best_seed = None
    best_split_dict = {}

    print(f"Simulating {iterations} random splits (3 equal sets) to minimize StdDev...")

    for seed in range(iterations):
        # 1. Shuffle indices deterministically
        rng = np.random.default_rng(seed)
        shuffled_indices = rng.permutation(n_projects)
        
        # 2. Split indices into 3 roughly equal chunks
        # np.array_split is fast enough on 1D integer arrays
        chunks_idx = np.array_split(shuffled_indices, 3)
        idx_train, idx_test, idx_val = chunks_idx[0], chunks_idx[1], chunks_idx[2]
        
        # 3. Sum counts for each set (Matrix Addition - Very Fast)
        # We sum the rows corresponding to the indices for each set
        counts_train = values_arr[idx_train].sum(axis=0)
        counts_test  = values_arr[idx_test].sum(axis=0)
        counts_val   = values_arr[idx_val].sum(axis=0)
        
        # Stack them for easy column-wise math: Shape (3, n_antipatterns)
        # Row 0=Train, 1=Test, 2=Val
        all_counts = np.vstack([counts_train, counts_test, counts_val])

        # If Training set has fewer than 8 examples
        if np.any(counts_train < 8):
            continue
            
        # If Test (row 1) or Val (row 2) has 0 examples
        if np.any(all_counts[1:] == 0):
            continue

        means = np.mean(all_counts, axis=0)
        stds = np.std(all_counts, axis=0)
        
        # Avoid division by zero if a mean is 0 (though constraints usually prevent this)
        means = np.where(means == 0, 1, means) 
        
        # Coefficient of Variation (CV) = Std / Mean
        cvs = stds / means
        
        # Total score is sum of squared CVs
        total_score = np.sum(cvs ** 2)

        if total_score < best_score:
            best_score = total_score
            best_seed = seed
            # We only store the strings when we find a new best to save memory
            best_split_dict = {
                'train': projects_arr[idx_train],
                'test': projects_arr[idx_test],
                'val': projects_arr[idx_val]
            }

    return best_seed, best_split_dict, best_score

best_seed, split_data, score = optimize_split(filtered_ground_truth_df)

print(f"\n--- OPTIMIZATION COMPLETE ---")
print(f"Best Seed Found: {best_seed}")
print(f"Balance Score (Lower is better): {score:.4f}")

def print_dist(name, projs):
    print(f"\n--- {name} Set ({len(projs)} Projects) ---")
    # We can use the original dataframe here for verification display
    subset = filtered_ground_truth_df[filtered_ground_truth_df['Project'].isin(projs)]
    counts = subset[subset['Antipattern'].isin(included_antipatterns)]['Antipattern'].value_counts()
    # Reindex to ensure order and show 0s
    print(counts.reindex(included_antipatterns, fill_value=0))

print_dist("Training", split_data['train'])
print_dist("Test", split_data['test'])
print_dist("Validation", split_data['val'])

# --- EXPORT ---
train_projects_final = split_data['train']
test_projects_final = split_data['test']
val_projects_final = split_data['val']

Pre-calculating project matrix...
Simulating 1000000 random splits (3 equal sets) to minimize StdDev...

--- OPTIMIZATION COMPLETE ---
Best Seed Found: 767573
Balance Score (Lower is better): 0.5168

--- Training Set (21 Projects) ---
Antipattern
Implicit Columns            175
ID Required                  55
Keyless Entry                30
Beware of the Unknown        23
Poor Man’s Search Engine     19
Rounding Errors              13
31 Flavors                   17
Name: count, dtype: int64

--- Test Set (20 Projects) ---
Antipattern
Implicit Columns            270
ID Required                 105
Keyless Entry                36
Beware of the Unknown        36
Poor Man’s Search Engine     21
Rounding Errors              16
31 Flavors                   39
Name: count, dtype: int64

--- Validation Set (20 Projects) ---
Antipattern
Implicit Columns            350
ID Required                  99
Keyless Entry                56
Beware of the Unknown        36
Poor Man’s Search Engine     15

In [22]:
training_set_df = filtered_ground_truth_df[filtered_ground_truth_df["Project"].isin(train_projects_final)]
training_set_df.to_csv('../datasets/training-set.csv', index=False)

In [23]:
test_set_df = filtered_ground_truth_df[filtered_ground_truth_df["Project"].isin(test_projects_final)]
test_set_df.to_csv('../datasets/test-set.csv', index=False)

In [24]:
validation_set_df = filtered_ground_truth_df[filtered_ground_truth_df["Project"].isin(val_projects_final)]
validation_set_df.to_csv('../datasets/validation-set.csv', index=False)